# Super basic recommendation system

This notebook implements a **hybrid recommendation model** combining **categorical metadata** (genres, themes, demographics) and **semantic narrative similarity** (synopsis text matching) on the anime database. It also generates a **network graph** dataset (nodes and edges) where node sizing represents popularity (`members`)

### Features:
1. **Environment Detection**: Works seamlessly on both **Kaggle** (utilizing GPUs/CPUs) and **local environments**.
2. **Categorical Features**: Multi-hot encodes genres, themes, and demographics to capture structural similarities.
3. **Semantic Features**: Uses **Sentence Transformers** (`all-MiniLM-L6-v2`) to embed synopsis text, with a fast statistical fallback to **TF-IDF** if packages or internet are unavailable.
4. **Hybrid Similarity**: Merges categorical and semantic similarity with a adjustable weight parameter $\alpha$.

### Step 1: Install Dependencies & Import Libraries
We check for `sentence-transformers` and install it if it's missing (highly recommended for deep semantic analysis of the synopses).

In [1]:
# Try to import sentence-transformers, install if missing
try:
    import sentence_transformers
    print("sentence-transformers is already installed.")
except ImportError:
    print("sentence-transformers not found. Installing...")
    !pip install -q sentence-transformers

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentence-transformers is already installed.


### Step 2: Configuration & Data Loading
We detect the running environment and seek the dataset in common paths.

In [14]:
ALPHA = 0.5 # Categorical similarity weight so semantic will be 1 - alpha.
USE_SENTENCE_TRANSFORMERS = True  # False = TF-IDF (for CPU)
K_NEIGHBORS = 20                  # no. of edges (similar connection) per anime
SIMILARITY_THRESHOLD = 0.15  # Minimum similarity score required to create an edge

paths = [
    "../datasets/anime_db_enriched_compact.json",
    "datasets/anime_db_enriched_compact.json",
    "./anime_db_enriched_compact.json",
    "/kaggle/input/datasets/avigyanray/anime-db-enriched-compact/anime_db_enriched_compact.json"
]

data_path = None
for path in paths:
    if os.path.exists(path):
        data_path = path
        break

if not data_path:
    raise FileNotFoundError("Couldnt find")

print(f"Loading data from: {data_path}")
with open(data_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f"Successfully loaded {len(df)} anime entries.")

Loading data from: /kaggle/input/datasets/avigyanray/anime-db-enriched-compact/anime_db_enriched_compact.json
Successfully loaded 29765 anime entries.


### Step 3: Data Preprocessing & Cleaning
Fill missing columns, clean list values, and handle numerical attributes for ranking and node sizing.

In [3]:
# Normalize list fields
df['genres'] = df['genres'].apply(lambda x: x if isinstance(x, list) else [])
df['themes'] = df['themes'].apply(lambda x: x if isinstance(x, list) else [])
df['demographics'] = df['demographics'].apply(lambda x: x if isinstance(x, list) else [])
df['studios'] = df['studios'].apply(lambda x: x if isinstance(x, list) else [])

# Handle text columns
df['synopsis'] = df['synopsis'].fillna("")
df['title_default'] = df['title_default'].fillna("Unknown Title")
df['title_en'] = df['title_en'].fillna(df['title_default'])

# Fill numeric missing fields
df['members'] = df['members'].fillna(0).astype(int)
df['favorites'] = df['favorites'].fillna(0).astype(int)
df['scored_avg'] = df['scored_avg'].fillna(0.0).astype(float)
df['sfw'] = df['sfw'].fillna(True).infer_objects(copy=False).astype(bool)
df['anime_id'] = df['anime_id'].astype(int)

print("Sample record preview:")
df[['anime_id', 'title_default', 'genres', 'themes', 'members', 'scored_avg']].head(20)

Sample record preview:


/tmp/ipykernel_150/2643926150.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sfw'] = df['sfw'].fillna(True).infer_objects(copy=False).astype(bool)


,anime_id,title_default,genres,themes,members,scored_avg
0,33840,Kabuki-bu!,[Comedy],"[acting, angst, asia, based on a light novel, ...",21263,6.60
1,59647,Eunbikkabiui Yesnal Yesjeog-e,[Fantasy],"[fantasy, historical, kids]",91,0.00
2,45496,Baobao Bashi: Qimiao Shuxue Da Maoxian,[Fantasy],"[anthropomorphic, fantasy, kids]",92,0.00
3,35022,Yuuki Aru Hotaru to Tobe Nai Hotaru,[Drama],"[drama, shorts]",259,0.00
4,56503,Atomic War Daichou-hen,[Avant Garde],[avant garde],139,0.00
5,43537,Lao Shu Jia Nu,[],"[animal protagonists, anthropomorphic, chinese...",141,0.00
6,35799,Miniforce,[],"[action, animal protagonists, anthropomorphic,...",363,5.71
7,55196,Wei Jiang,"[Action, Fantasy]","[action, cg animation, cg-anime, chinese anima...",234,0.00
8,16518,Arata Kangatari,"[Adventure, Fantasy]","[action, adventure, alternative world, angst, ...",75750,6.42
9,38544,Egao no Daika,"[Drama, Fantasy, Sci-Fi]","[3d cg animation, action, action drama, alcoho...",40249,6.03


### Step 4: Categorical Feature Matrix
Applying multi hot encoding after combining the tags

In [4]:
# Combine categorical attributes into a list of tags
def combine_tags(row):
    tags = set()
    tags.update(row['genres'])
    tags.update(row['themes'])
    tags.update(row['demographics'])
    # tags.update([f"studio_{s.lower()}" for s in row['studios']])
    return list(tags)

df['meta_tags'] = df.apply(combine_tags, axis=1)

# Fit MultiLabelBinarizer
mlb = MultiLabelBinarizer(sparse_output=True)
categorical_matrix = mlb.fit_transform(df['meta_tags'])

print(f"Total anime: {len(df)}")
print(f"Total unique tags: {len(mlb.classes_)}")
print(f"Categorical feature shape: {categorical_matrix.shape}")

print(f"Average tags per anime: {df['meta_tags'].apply(len).mean():.2f}")
print(f"Max tags on an anime: {df['meta_tags'].apply(len).max()}")
print(f"Min tags on an anime: {df['meta_tags'].apply(len).min()}")

print("Computing categorical cosine similarity...")
cat_sim = cosine_similarity(categorical_matrix, dense_output=True)
print("Categorical similarity computation complete.")

Total anime: 29765
Total unique tags: 3033
Categorical feature shape: (29765, 3033)
Average tags per anime: 19.40
Max tags on an anime: 212
Min tags on an anime: 0
Computing categorical cosine similarity...
Categorical similarity computation complete.


### Step 5: Semantic Feature Matrix

In [5]:
sem_sim = None

if USE_SENTENCE_TRANSFORMERS:
    try:
        from sentence_transformers import SentenceTransformer
        import torch
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading sentence-transformer model ('all-MiniLM-L6-v2') on {device.upper()}...")
        model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        
        # Clean synopsis list
        synopses = df['synopsis'].tolist()
        
        print("Encoding synopses (this may take a few minutes depending on hardware)...")
        embeddings = model.encode(synopses, show_progress_bar=True, batch_size=128)
        
        print("Computing semantic cosine similarity...")
        sem_sim = cosine_similarity(embeddings)
        
    except Exception as e:
        print(f"Sentence Transformers failed/not available: {e}")
        print("Switching fallback to TF-IDF...")
        USE_SENTENCE_TRANSFORMERS = False

if not USE_SENTENCE_TRANSFORMERS:
    print("Computing TF-IDF matrices on synopses...")
    tfidf = TfidfVectorizer(max_features=15000, stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df['synopsis'])
    
    print("Computing semantic cosine similarity (TF-IDF)...")
    sem_sim = cosine_similarity(tfidf_matrix, dense_output=True)

print("Semantic similarity computation complete.")

Loading sentence-transformer model ('all-MiniLM-L6-v2') on CUDA...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding synopses (this may take a few minutes depending on hardware)...


Batches:   0%|          | 0/233 [00:00<?, ?it/s]

Computing semantic cosine similarity...
Semantic similarity computation complete.


### Step 6: Compute Hybrid Similarity Matrix
Blend the two similarity spaces using the linear combination formula:
$$\text{Similarity}_{\text{hybrid}} = \alpha \cdot \text{Similarity}_{\text{categorical}} + (1 - \alpha) \cdot \text{Similarity}_{\text{semantic}}$$

In [15]:
print(f"Combining similarities (Alpha: {ALPHA} Categorical / {1.0 - ALPHA} Semantic)...")
hybrid_sim = ALPHA * cat_sim + (1 - ALPHA) * sem_sim

# Set self-similarity diagonal to 0 to avoid recommending the item itself
np.fill_diagonal(hybrid_sim, 0.0)
print("Hybrid similarity matrix constructed.")

Combining similarities (Alpha: 0.5 Categorical / 0.5 Semantic)...
Hybrid similarity matrix constructed.


### Step 7: Build Query Recommendation Searcher
Create a validation function to retrieve top recommendations for any given anime title.

In [ ]:
def search_anime(query):
    """Helper to search anime index, titles, or anime_id"""
    try:
        query_id = int(query)
        matches = df[df['anime_id'] == query_id]
    except (ValueError, TypeError):
        matches = pd.DataFrame()
        
    if matches.empty:
        query_str = str(query)
        matches = df[df['title_default'].str.contains(query_str, case=False, na=False) |
                     df['title_en'].str.contains(query_str, case=False, na=False)]
    return matches[['anime_id', 'title_default', 'title_en', 'type', 'scored_avg', 'members']]

def recommend_hybrid(query_id_or_title, top_n=10, sfw_only=True, popularity_weight=0.15):
    """
    Recommend anime using a hybrid similarity score blended with log-popularity.
    
    popularity_weight: weight factor (0.0 to 1.0) given to popularity (members).
                       0.0 means recommendations are based purely on content similarity.
                       1.0 means pure popularity ranking.
    """
    # Find matching row
    try:
        query_id = int(query_id_or_title)
        matches = df[df['anime_id'] == query_id]
    except (ValueError, TypeError):
        matches = pd.DataFrame()
        
    if matches.empty:
        query_str = str(query_id_or_title)
        matches = df[df['title_default'].str.contains(query_str, case=False, na=False) |
                     df['title_en'].str.contains(query_str, case=False, na=False)]
        
    if matches.empty:
        print(f"No anime matching '{query_id_or_title}' found.")
        return None
        
    # Get the index of the first match
    idx = matches.index[0]
    query_row = df.iloc[idx]
    
    print(f"Showing recommendations for: {query_row['title_default']} (ID: {query_row['anime_id']}, Type: {query_row['type']})")
    print(f"Genres: {query_row['genres']} | Themes: {query_row['themes']}")
    print("=" * 110)
    
    # 1. Get raw content similarity scores
    similarity_scores = hybrid_sim[idx].copy()
    
    # 2. Compute log-scaled popularity score (0.0 to 1.0)
    log_members = np.log1p(df['members'].values)
    max_log_members = log_members.max() if log_members.max() > 0 else 1.0
    popularity_scores = log_members / max_log_members
    
    # 3. Combine similarity and popularity
    combined_scores = (1.0 - popularity_weight) * similarity_scores + popularity_weight * popularity_scores
    
    # Sort indices by combined score descending
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    results = []
    count = 0
    
    for target_idx in sorted_indices:
        if target_idx == idx:
            continue
            
        row = df.iloc[target_idx]
        
        # Filters
        if sfw_only and not row['sfw']:
            continue
            
        results.append({
            "Rank": count + 1,
            "ID": int(row['anime_id']),
            "Title": row['title_default'],
            "Type": row['type'],
            "Score": row['scored_avg'],
            "Members": row['members'],
            "Similarity Score": similarity_scores[target_idx],
            "Combined Score": combined_scores[target_idx],
            "Genres": ", ".join(row['genres']),
            "Synopsis": row['synopsis'][:140] + "..." if row['synopsis'] else ""
        })
        
        count += 1
        if count >= top_n:
            break
            
    rec_df = pd.DataFrame(results)
    return rec_df

Let's test this model out with a title query (e.g., search for something in the dataset like 'Kabuki' or 'Fruits Basket'):

In [17]:
# Search helper
search_anime("grand blue")

,anime_id,title_default,title_en,type,scored_avg
12270,62542,Grand Blue Season 3,Grand Blue Dreaming Season 3,TV,NaN
21388,37105,Grand Blue,Grand Blue Dreaming,TV,8.45
21796,41574,Guraburu!,Grand Blues!,TV,5.80
26937,59986,Grand Blue Season 2,Grand Blue Dreaming Season 2,TV,8.50


In [ ]:
# Get recommendations
recommend_hybrid("Grand Blue", top_n=20)

Showing recommendations for: Grand Blue Season 3 (TV)
Genres: ['Comedy'] | Themes: ['adult cast', 'alcohol', 'based on a manga', 'club', 'college', 'comedy', 'ecchi', 'gag', 'gag humor', 'japanese production', 'mature romance', 'nonsense-comedy', 'ocean', 'otaku', 'present', 'romance', 'scuba diving', 'seinen', 'sequel', 'slapstick', 'slice of life', 'sports', 'university', 'water sports']


,Rank,Title,Type,Score,Members,Match Score,Genres,Synopsis
0,1,Grand Blue Season 2,TV,8.50,175364,0.503475,Comedy,Three months have passed since Iori Kitahara b...
1,2,Genshiken Nidaime Specials,Special,6.67,8447,0.485624,Comedy,Special episodes included in BD....
2,3,Osomatsu-san 3rd Season,TV,7.39,25220,0.465720,Comedy,Third season of Osomatsu-san....
3,4,Bernard-jou Iwaku.: Ofuro Dokusho,Special,5.71,1999,0.461786,Comedy,Special episode included with BD/DVD release....
4,5,Recorder to Randoseru Re♪,TV,6.79,31528,0.457756,Comedy,Second season....
5,6,Grand Blue,TV,8.45,882888,0.456742,Comedy,Iori Kitahara moves to the coastal town of Izu...
6,7,Osomatsu-san 4th Season,TV,6.16,6280,0.449381,Comedy,Fourth season of Osomatsu-san....
7,8,Gurazeni Season 2,TV,6.74,5212,0.447983,Sports,Second season of Gurazeni....
8,9,Akachan Honbuchou 3rd Season,TV,NaN,624,0.445888,Comedy,The third season of Akachan Honbuchou....
9,10,Teekyuu 3,TV,6.73,28738,0.443720,Comedy,Third season of Teekyuu series....


### Step 7.5: Advanced Model Improvements
We can further improve recommendations by:
1. **Quality Boosting**: Blend similarity scores with the MyAnimeList user rating (`scored_avg`) so highly-rated shows are favored over low-rated ones.
2. **Franchise / Sequel Filter**: Demote or exclude sequel seasons (like *Grand Blue Season 2* for *Grand Blue*) to give fresh new anime recommendations.

In [ ]:
def recommend_advanced(query_id_or_title, top_n=10, sfw_only=True, 
                       popularity_weight=0.1, quality_weight=0.1,
                       exclude_sequels=False):
    """
    Advanced recommendation function incorporating:
    - Categorical & Semantic similarity
    - Popularity log-scale boost
    - Quality (average score) boost
    - Optional sequel / franchise filter
    """
    # 1. Find matching row
    try:
        query_id = int(query_id_or_title)
        matches = df[df['anime_id'] == query_id]
    except (ValueError, TypeError):
        matches = pd.DataFrame()
        
    if matches.empty:
        query_str = str(query_id_or_title)
        matches = df[df['title_default'].str.contains(query_str, case=False, na=False) |
                     df['title_en'].str.contains(query_str, case=False, na=False)]
        
    if matches.empty:
        print(f"No anime matching '{query_id_or_title}' found.")
        return None
        
    idx = matches.index[0]
    query_row = df.iloc[idx]
    query_title = query_row['title_default'].lower()
    
    print(f"Showing advanced recommendations for: {query_row['title_default']} (Score: {query_row['scored_avg']})")
    print("=" * 110)
    
    # 2. Content similarity
    sim_scores = hybrid_sim[idx].copy()
    
    # 3. Popularity score (0.0 to 1.0)
    log_members = np.log1p(df['members'].values)
    popularity_scores = log_members / (log_members.max() if log_members.max() > 0 else 1.0)
    
    # 4. Quality score (0.0 to 1.0, since MAL is 1-10 scale)
    quality_scores = df['scored_avg'].values / 10.0
    
    # 5. Combine scores
    content_weight = 1.0 - popularity_weight - quality_weight
    combined_scores = (content_weight * sim_scores + 
                       popularity_weight * popularity_scores + 
                       quality_weight * quality_scores)
    
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    results = []
    count = 0
    
    for target_idx in sorted_indices:
        if target_idx == idx:
            continue
            
        row = df.iloc[target_idx]
        
        # Filters
        if sfw_only and not row['sfw']:
            continue
            
        # Optional Sequel Filter
        if exclude_sequels:
            is_sequel = 'sequel' in row['themes']
            title_lower = row['title_default'].lower()
            title_en_lower = row['title_en'].lower()
            is_franchise_overlap = (
                (query_title in title_lower or title_lower in query_title) or
                (query_row['title_en'].lower() in title_en_lower or title_en_lower in query_row['title_en'].lower())
            )
            if is_sequel or (is_franchise_overlap and abs(len(title_lower) - len(query_title)) > 2):
                continue
                
        results.append({
            "Rank": count + 1,
            "ID": int(row['anime_id']),
            "Title": row['title_default'],
            "Type": row['type'],
            "Score": row['scored_avg'],
            "Members": row['members'],
            "Sim": sim_scores[target_idx],
            "Combined": combined_scores[target_idx],
            "Genres": ", ".join(row['genres']),
            "Synopsis": row['synopsis'][:140] + "..." if row['synopsis'] else ""
        })
        
        count += 1
        if count >= top_n:
            break
            
    return pd.DataFrame(results)

In [ ]:
# Test advanced recommendation (e.g. search using title or ID, with popularity & quality boost, and franchise exclusion)
recommend_advanced("Grand Blue", top_n=10, popularity_weight=0.15, quality_weight=0.15, exclude_sequels=True)

### Step 8: Build the Graph (Nodes and Edges Export)
Here we create the similarity graph structures:
1. **Nodes list**: Represents the anime items, storing attributes like Title, Rating, Type, Genres, and Popularity (`members`).
2. **Edges list**: Represents similar links. We loop in batches to be memory friendly, selecting the top $K$ nearest neighbors for each anime. We also enforce a lower similarity threshold to prune poor/unrelated connections.

In [10]:
nodes = []
edges = []
num_anime = len(df)

# Create nodes list
for idx, row in df.iterrows():
    nodes.append({
        "id": int(row['anime_id']),
        "title": str(row['title_default']),
        "title_en": str(row['title_en']),
        "type": str(row['type']),
        "scored_avg": float(row['scored_avg']),
        "members": int(row['members']),
        "favorites": int(row['favorites']),
        "genres": row['genres'],
        "sfw": bool(row['sfw']),
        "image_url": str(row['image_url'])
    })

print(f"Generated nodes meta list ({len(nodes)} nodes).")

# Batch extract top K edges per node
print("Generating network edges (top K nearest neighbors per anime)...")
batch_size = 2000

for start_idx in range(0, num_anime, batch_size):
    end_idx = min(start_idx + batch_size, num_anime)
    sim_batch = hybrid_sim[start_idx:end_idx]
    
    # Use argpartition to find top K indices along columns efficiently
    top_k_batch = np.argpartition(sim_batch, -K_NEIGHBORS, axis=1)[:, -K_NEIGHBORS:]
    
    for offset, top_k in enumerate(top_k_batch):
        current_idx = start_idx + offset
        current_id = int(df.iloc[current_idx]['anime_id'])
        
        # Sort these K indices by their actual similarity weight descending
        actual_scores = sim_batch[offset, top_k]
        sorted_order = np.argsort(actual_scores)[::-1]
        sorted_top_k = top_k[sorted_order]
        
        for target_idx in sorted_top_k:
            weight = float(hybrid_sim[current_idx, target_idx])
            if weight >= SIMILARITY_THRESHOLD:
                edges.append({
                    "source": current_id,
                    "target": int(df.iloc[target_idx]['anime_id']),
                    "weight": weight
                })

print(f"Generated {len(edges)} weighted edges.")

Generated nodes meta list (29765 nodes).
Generating network edges (top K nearest neighbors per anime)...
Generated 595184 weighted edges.


### Step 9: Save Graph Files
Save the nodes and edges as CSVs (for Gephi imports) and a single JSON file (for frontend JS like D3 or Cytoscape).

In [11]:
# 1. Save as complete JSON
graph_json = {
    "nodes": nodes,
    "edges": edges
}
with open("anime_similarity_graph.json", "w", encoding="utf-8") as f:
    json.dump(graph_json, f, indent=2)
print("Saved complete graph to 'anime_similarity_graph.json'")

# 2. Save as separate CSVS
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)

nodes_df.to_csv("anime_nodes.csv", index=False)
edges_df.to_csv("anime_edges.csv", index=False)

print("Saved 'anime_nodes.csv' and 'anime_edges.csv'.")

Saved complete graph to 'anime_similarity_graph.json'
Saved 'anime_nodes.csv' and 'anime_edges.csv'.
